# Exercise 3 #

### List the files in folder ###

In [95]:
import glob

# extract list of video files
extensions = ['mp4', 'avi', 'mov', 'mkv', 'flv']

video_files = []
for ext in extensions:
    video_files.extend(glob.glob(f"films/*.{ext}"))

print(video_files)

['films/Voyage_to_the_Planet_of_Prehistoric_Women.mp4', 'films/Cosmos_War_of_the_Planets.mp4', 'films/The_Hill_Gang_Rides_Again.mp4', 'films/The_Gun_and_the_Pulpit.avi', 'films/Last_man_on_earth_1964.mov']


### Extract Metadata from videos ###

In [97]:
import subprocess
import json
import os
import pandas as pd
from IPython.display import display

def get_video_metadata(video_path):
    # Extract metadata from a video file using ffprobe
    cmd = [
        "ffprobe",
        "-v", "error",
        "-show_entries", "format:stream",
        "-of", "json",
        video_path
    ]
    
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    
    if result.returncode != 0:
        print(f"Error processing {video_path}: {result.stderr}")
        return None
    
    return json.loads(result.stdout)

def format_frame_rate(frame_rate):
    # Convert frame rate from fraction to FPS
    try:
        num, den = map(int, frame_rate.split('/'))
        return f"{num / den:.2f} FPS"
    except:
        return "N/A"

def format_bit_rate(bit_rate, unit="Mb"):
    # Convert bit rate from bits per second to Mb/s or kb/s
    try:
        bit_rate = int(bit_rate)
        if unit == "Mb":
            return f"{bit_rate / 1_000_000:.2f} Mb/s"
        elif unit == "kb":
            return f"{bit_rate / 1_000:.2f} kb/s"
    except:
        return "N/A"

def extract_metadata_from_folder(folder_path):
    # Extract relevant metadata from all video files in the given folder
    data = []
    
    for file in os.listdir(folder_path):
        video_path = os.path.join(folder_path, file)
        if os.path.isfile(video_path):
            metadata = get_video_metadata(video_path)
            if metadata:
                format_info = metadata.get("format", {})
                streams = metadata.get("streams", [])

                # Extract video and audio stream details
                video_stream = next((s for s in streams if s.get("codec_type") == "video"), {})
                audio_stream = next((s for s in streams if s.get("codec_type") == "audio"), {})

                width = video_stream.get("width", None)
                height = video_stream.get("height", None)
                aspect_ratio = video_stream.get("display_aspect_ratio", "N/A")

                # Compute aspect ratio manually if missing
                if aspect_ratio == "N/A" and width and height:
                    aspect_ratio = f"{width}:{height}"

                # Extract the exact container format, fallback to file extension if needed
                detected_container = format_info.get("format_name", "N/A").split(",")[0]
                file_extension = file.split(".")[-1].lower()

                # If detected container is ambiguous, use file extension
                if detected_container == "mov":
                    container = file_extension
                else:
                    container = detected_container

                data.append({
                    "File Name": file,
                    "Container": container,
                    "Video Codec": video_stream.get("codec_name", "N/A"),
                    "Audio Codec": audio_stream.get("codec_name", "N/A"),
                    "Frame Rate": format_frame_rate(video_stream.get("avg_frame_rate", "N/A")),
                    "Aspect Ratio": aspect_ratio,
                    "Resolution": f"{width}x{height}" if width and height else "N/A",
                    "Bit Rate (Video)": format_bit_rate(video_stream.get("bit_rate", "N/A"), "Mb"),
                    "Bit Rate (Audio)": format_bit_rate(audio_stream.get("bit_rate", "N/A"), "kb"),
                    "Channels": "Stereo" if audio_stream.get("channels", 0) == 2 else audio_stream.get("channels", "N/A"),
                })
    
    return pd.DataFrame(data)

# Extract metadata and display in table format
films_metadata_df = extract_metadata_from_folder("films")

# Display the table in Jupyter Notebook
display(films_metadata_df)

Error processing films/.DS_Store: films/.DS_Store: Invalid data found when processing input



,File Name,Container,Video Codec,Audio Codec,Frame Rate,Aspect Ratio,Resolution,Bit Rate (Video),Bit Rate (Audio),Channels
0,Last_man_on_earth_1964.mov,mov,prores,pcm_s16le,23.98 FPS,16:9,640x360,9.29 Mb/s,1536.00 kb/s,Stereo
1,Voyage_to_the_Planet_of_Prehistoric_Women.mp4,mp4,hevc,mp3,29.97 FPS,16:9,640x360,8.04 Mb/s,320.00 kb/s,Stereo
2,The_Gun_and_the_Pulpit.avi,avi,rawvideo,pcm_s16le,25.00 FPS,720:404,720x404,87.44 Mb/s,1536.00 kb/s,Stereo
3,Cosmos_War_of_the_Planets.mp4,mp4,h264,aac,29.97 FPS,314:177,628x354,2.99 Mb/s,317.10 kb/s,Stereo
4,The_Hill_Gang_Rides_Again.mp4,mp4,h264,aac,25.00 FPS,16:9,640x360,7.54 Mb/s,253.27 kb/s,Stereo


### Filter Out Unfulfilled Format ###

In [101]:
import subprocess
import json
import os
import pandas as pd
from IPython.display import display

# Festival's required format
REQUIRED_FORMAT = {
    "Container": "mp4",
    "Video Codec": "h264",
    "Audio Codec": "aac",
    "Frame Rate": "25.00 FPS",
    "Aspect Ratio": "16:9",
    "Resolution": "640x360",
    "Bit Rate (Video) Min": 2.0,  # Mb/s
    "Bit Rate (Video) Max": 5.0,  # Mb/s
    "Bit Rate (Audio) Max": 256.0,  # kb/s
    "Channels": "Stereo"
}

def get_video_metadata(video_path):
    # Extract metadata from a video file using ffprobe
    cmd = [
        "ffprobe",
        "-v", "error",
        "-show_entries", "format:stream",
        "-of", "json",
        video_path
    ]
    
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    
    if result.returncode != 0:
        print(f"Error processing {video_path}: {result.stderr}")
        return None
    
    return json.loads(result.stdout)

def format_frame_rate(frame_rate):
    # Convert frame rate from fraction to FPS
    try:
        num, den = map(int, frame_rate.split('/'))
        return f"{num / den:.2f} FPS"
    except:
        return "N/A"

def format_bit_rate(bit_rate, unit="Mb"):
    # Convert bit rate from bits per second to Mb/s or kb/s
    try:
        bit_rate = int(bit_rate)
        if unit == "Mb":
            return round(bit_rate / 1_000_000, 2)  # Convert to Mb/s
        elif unit == "kb":
            return round(bit_rate / 1_000, 2)  # Convert to kb/s
    except:
        return "N/A"

def extract_metadata_from_folder(folder_path):
    # Extract relevant metadata from all video files in the given folder
    data = []
    
    for file in os.listdir(folder_path):
        video_path = os.path.join(folder_path, file)
        if os.path.isfile(video_path):
            metadata = get_video_metadata(video_path)
            if metadata:
                format_info = metadata.get("format", {})
                streams = metadata.get("streams", [])

                # Extract video and audio stream details
                video_stream = next((s for s in streams if s.get("codec_type") == "video"), {})
                audio_stream = next((s for s in streams if s.get("codec_type") == "audio"), {})

                width = video_stream.get("width", None)
                height = video_stream.get("height", None)
                aspect_ratio = video_stream.get("display_aspect_ratio", "N/A")

                # Compute aspect ratio manually if missing
                if aspect_ratio == "N/A" and width and height:
                    aspect_ratio = f"{width}:{height}"

                # Extract the exact container format, fallback to file extension if needed
                detected_container = format_info.get("format_name", "N/A").split(",")[0]
                file_extension = file.split(".")[-1].lower()

                # If detected container is ambiguous, use file extension
                container = file_extension if detected_container == "mov" else detected_container

                # Store extracted metadata
                data.append({
                    "File Name": file,
                    "Container": container,
                    "Video Codec": video_stream.get("codec_name", "N/A"),
                    "Audio Codec": audio_stream.get("codec_name", "N/A"),
                    "Frame Rate": format_frame_rate(video_stream.get("avg_frame_rate", "N/A")),
                    "Aspect Ratio": aspect_ratio,
                    "Resolution": f"{width}x{height}" if width and height else "N/A",
                    "Bit Rate (Video)": format_bit_rate(video_stream.get("bit_rate", "N/A"), "Mb"),
                    "Bit Rate (Audio)": format_bit_rate(audio_stream.get("bit_rate", "N/A"), "kb"),
                    "Channels": "Stereo" if audio_stream.get("channels", 0) == 2 else audio_stream.get("channels", "N/A"),
                })
    
    return pd.DataFrame(data)

def check_compliance_and_generate_report(films_metadata_df, report_file="format_compliance_report.txt"):
    # Check each film against the required format and generate a report for non-compliant films
    non_compliant_films = []
    
    for _, row in films_metadata_df.iterrows():
        issues = []

        # Check each format requirement
        if row["Container"] != REQUIRED_FORMAT["Container"]:
            issues.append(f"Container should be {REQUIRED_FORMAT['Container']}, found {row['Container']}")
        
        if row["Video Codec"] != REQUIRED_FORMAT["Video Codec"]:
            issues.append(f"Video Codec should be {REQUIRED_FORMAT['Video Codec']}, found {row['Video Codec']}")
        
        if row["Audio Codec"] != REQUIRED_FORMAT["Audio Codec"]:
            issues.append(f"Audio Codec should be {REQUIRED_FORMAT['Audio Codec']}, found {row['Audio Codec']}")
        
        if row["Frame Rate"] != REQUIRED_FORMAT["Frame Rate"]:
            issues.append(f"Frame Rate should be {REQUIRED_FORMAT['Frame Rate']}, found {row['Frame Rate']}")
        
        if row["Aspect Ratio"] != REQUIRED_FORMAT["Aspect Ratio"]:
            issues.append(f"Aspect Ratio should be {REQUIRED_FORMAT['Aspect Ratio']}, found {row['Aspect Ratio']}")
        
        if row["Resolution"] != REQUIRED_FORMAT["Resolution"]:
            issues.append(f"Resolution should be {REQUIRED_FORMAT['Resolution']}, found {row['Resolution']}")
        
        if not (REQUIRED_FORMAT["Bit Rate (Video) Min"] <= row["Bit Rate (Video)"] <= REQUIRED_FORMAT["Bit Rate (Video) Max"]):
            issues.append(f"Video Bit Rate should be between {REQUIRED_FORMAT['Bit Rate (Video) Min']} - {REQUIRED_FORMAT['Bit Rate (Video) Max']} Mb/s, found {row['Bit Rate (Video)']} Mb/s")
        
        if row["Bit Rate (Audio)"] > REQUIRED_FORMAT["Bit Rate (Audio) Max"]:
            issues.append(f"Audio Bit Rate should be up to {REQUIRED_FORMAT['Bit Rate (Audio) Max']} kb/s, found {row['Bit Rate (Audio)']} kb/s")
        
        if row["Channels"] != REQUIRED_FORMAT["Channels"]:
            issues.append(f"Audio Channels should be {REQUIRED_FORMAT['Channels']}, found {row['Channels']}")

        # If there are any issues, add them to the report
        if issues:
            non_compliant_films.append(f"Film: {row['File Name']}\n" + "\n".join(issues) + "\n")

    # Save report
    with open(report_file, "w") as f:
        if non_compliant_films:
            f.write("Non-Compliant Films Report:\n\n" + "\n".join(non_compliant_films))
        else:
            f.write("All films meet the required format.")

    print(f"Report generated: {report_file}")

# Extract metadata
films_metadata_df = extract_metadata_from_folder("films")

# Display metadata table in Jupyter Notebook
display(films_metadata_df)

# Check compliance and generate report
check_compliance_and_generate_report(films_metadata_df)

Error processing films/.DS_Store: films/.DS_Store: Invalid data found when processing input



,File Name,Container,Video Codec,Audio Codec,Frame Rate,Aspect Ratio,Resolution,Bit Rate (Video),Bit Rate (Audio),Channels
0,Last_man_on_earth_1964.mov,mov,prores,pcm_s16le,23.98 FPS,16:9,640x360,9.29,1536.00,Stereo
1,Voyage_to_the_Planet_of_Prehistoric_Women.mp4,mp4,hevc,mp3,29.97 FPS,16:9,640x360,8.04,320.00,Stereo
2,The_Gun_and_the_Pulpit.avi,avi,rawvideo,pcm_s16le,25.00 FPS,720:404,720x404,87.44,1536.00,Stereo
3,Cosmos_War_of_the_Planets.mp4,mp4,h264,aac,29.97 FPS,314:177,628x354,2.99,317.10,Stereo
4,The_Hill_Gang_Rides_Again.mp4,mp4,h264,aac,25.00 FPS,16:9,640x360,7.54,253.27,Stereo


Report generated: format_compliance_report.txt


### Automatically Convert to Correct Formats ###

In [105]:
import subprocess

def convert_non_compliant_films(films_metadata_df, folder_path):
    # Convert non-compliant films to the correct format using ffmpeg
    
    for _, row in films_metadata_df.iterrows():
        issues = []

        # Check if the film is non-compliant
        if row["Container"] != REQUIRED_FORMAT["Container"]:
            issues.append("Container format")
        if row["Video Codec"] != REQUIRED_FORMAT["Video Codec"]:
            issues.append("Video Codec")
        if row["Audio Codec"] != REQUIRED_FORMAT["Audio Codec"]:
            issues.append("Audio Codec")
        if row["Frame Rate"] != REQUIRED_FORMAT["Frame Rate"]:
            issues.append("Frame Rate")
        if row["Aspect Ratio"] != REQUIRED_FORMAT["Aspect Ratio"]:
            issues.append("Aspect Ratio")
        if row["Resolution"] != REQUIRED_FORMAT["Resolution"]:
            issues.append("Resolution")
        if not (REQUIRED_FORMAT["Bit Rate (Video) Min"] <= row["Bit Rate (Video)"] <= REQUIRED_FORMAT["Bit Rate (Video) Max"]):
            issues.append("Video Bit Rate")
        if row["Bit Rate (Audio)"] > REQUIRED_FORMAT["Bit Rate (Audio) Max"]:
            issues.append("Audio Bit Rate")
        if row["Channels"] != REQUIRED_FORMAT["Channels"]:
            issues.append("Audio Channels")

        # If there are issues, convert the file
        if issues:
            input_file = os.path.join(folder_path, row["File Name"])
            output_file = os.path.join(folder_path, row["File Name"].rsplit(".", 1)[0] + "_formatOK.mp4")

            print(f"Converting: {row['File Name']} (Fixing: {', '.join(issues)}) → {output_file}")

            # ffmpeg command to convert the video
            cmd = [
                "ffmpeg",
                "-i", input_file,             # Input file
                "-c:v", "libx264",            # Video codec: H.264
                "-b:v", "3M",                 # Video bitrate: 3 Mb/s (within 2-5 Mb/s)
                "-vf", "scale=640:360",       # Resize to 640x360
                "-r", "25",                   # Frame rate: 25 FPS
                "-aspect", "16:9",            # Aspect ratio: 16:9
                "-c:a", "aac",                # Audio codec: AAC
                "-b:a", "256k",               # Audio bitrate: up to 256 kb/s
                "-ac", "2",                   # Audio channels: Stereo
                "-preset", "fast",            # Fast encoding preset
                "-y",                          # Overwrite output if exists
                output_file
            ]

            # Run the ffmpeg command
            subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

            print(f"Conversion completed: {output_file}")

# Run conversion for non-compliant films
convert_non_compliant_films(films_metadata_df, "films")

Converting: Last_man_on_earth_1964.mov (Fixing: Container format, Video Codec, Audio Codec, Frame Rate, Video Bit Rate, Audio Bit Rate) → films/Last_man_on_earth_1964_formatOK.mp4
Conversion completed: films/Last_man_on_earth_1964_formatOK.mp4
Converting: Voyage_to_the_Planet_of_Prehistoric_Women.mp4 (Fixing: Video Codec, Audio Codec, Frame Rate, Video Bit Rate, Audio Bit Rate) → films/Voyage_to_the_Planet_of_Prehistoric_Women_formatOK.mp4
Conversion completed: films/Voyage_to_the_Planet_of_Prehistoric_Women_formatOK.mp4
Converting: The_Gun_and_the_Pulpit.avi (Fixing: Container format, Video Codec, Audio Codec, Aspect Ratio, Resolution, Video Bit Rate, Audio Bit Rate) → films/The_Gun_and_the_Pulpit_formatOK.mp4
Conversion completed: films/The_Gun_and_the_Pulpit_formatOK.mp4
Converting: Cosmos_War_of_the_Planets.mp4 (Fixing: Frame Rate, Aspect Ratio, Resolution, Audio Bit Rate) → films/Cosmos_War_of_the_Planets_formatOK.mp4
Conversion completed: films/Cosmos_War_of_the_Planets_formatOK.

### Check all output files after conversion ###

In [109]:
import subprocess
import json
import os
import pandas as pd
from IPython.display import display

def get_video_metadata(video_path):
    # Extract metadata from a video file using ffprobe
    cmd = [
        "ffprobe",
        "-v", "error",
        "-show_entries", "format:stream",
        "-of", "json",
        video_path
    ]
    
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    
    if result.returncode != 0:
        print(f"Error processing {video_path}: {result.stderr}")
        return None
    
    return json.loads(result.stdout)

def format_frame_rate(frame_rate):
    # Convert frame rate from fraction to FPS
    try:
        num, den = map(int, frame_rate.split('/'))
        return f"{num / den:.2f} FPS"
    except:
        return "N/A"

def format_bit_rate(bit_rate, unit="Mb"):
    # Convert bit rate from bits per second to Mb/s or kb/s
    try:
        bit_rate = int(bit_rate)
        if unit == "Mb":
            return f"{bit_rate / 1_000_000:.2f} Mb/s"
        elif unit == "kb":
            return f"{bit_rate / 1_000:.2f} kb/s"
    except:
        return "N/A"

def extract_metadata_from_folder(folder_path):
    # Extract relevant metadata from all video files in the given folder
    data = []
    
    for file in os.listdir(folder_path):
        video_path = os.path.join(folder_path, file)
        if os.path.isfile(video_path):
            metadata = get_video_metadata(video_path)
            if metadata:
                format_info = metadata.get("format", {})
                streams = metadata.get("streams", [])

                # Extract video and audio stream details
                video_stream = next((s for s in streams if s.get("codec_type") == "video"), {})
                audio_stream = next((s for s in streams if s.get("codec_type") == "audio"), {})

                width = video_stream.get("width", None)
                height = video_stream.get("height", None)
                aspect_ratio = video_stream.get("display_aspect_ratio", "N/A")

                # Compute aspect ratio manually if missing
                if aspect_ratio == "N/A" and width and height:
                    aspect_ratio = f"{width}:{height}"

                # Extract the exact container format, fallback to file extension if needed
                detected_container = format_info.get("format_name", "N/A").split(",")[0]
                file_extension = file.split(".")[-1].lower()

                # If detected container is ambiguous, use file extension
                if detected_container == "mov":
                    container = file_extension
                else:
                    container = detected_container

                data.append({
                    "File Name": file,
                    "Container": container,
                    "Video Codec": video_stream.get("codec_name", "N/A"),
                    "Audio Codec": audio_stream.get("codec_name", "N/A"),
                    "Frame Rate": format_frame_rate(video_stream.get("avg_frame_rate", "N/A")),
                    "Aspect Ratio": aspect_ratio,
                    "Resolution": f"{width}x{height}" if width and height else "N/A",
                    "Bit Rate (Video)": format_bit_rate(video_stream.get("bit_rate", "N/A"), "Mb"),
                    "Bit Rate (Audio)": format_bit_rate(audio_stream.get("bit_rate", "N/A"), "kb"),
                    "Channels": "Stereo" if audio_stream.get("channels", 0) == 2 else audio_stream.get("channels", "N/A"),
                })
    
    return pd.DataFrame(data)

# Extract metadata and display in table format
films_metadata_df = extract_metadata_from_folder("films")

# Display the table in Jupyter Notebook
display(films_metadata_df)

Error processing films/.DS_Store: films/.DS_Store: Invalid data found when processing input



,File Name,Container,Video Codec,Audio Codec,Frame Rate,Aspect Ratio,Resolution,Bit Rate (Video),Bit Rate (Audio),Channels
0,Last_man_on_earth_1964_formatOK.mp4,mp4,h264,aac,25.00 FPS,16:9,640x360,3.13 Mb/s,240.47 kb/s,Stereo
1,Voyage_to_the_Planet_of_Prehistoric_Women_form...,mp4,h264,aac,25.00 FPS,16:9,640x360,2.88 Mb/s,242.37 kb/s,Stereo
2,Cosmos_War_of_the_Planets_formatOK.mp4,mp4,h264,aac,25.00 FPS,16:9,640x360,2.97 Mb/s,245.65 kb/s,Stereo
3,The_Hill_Gang_Rides_Again_formatOK.mp4,mp4,h264,aac,25.00 FPS,16:9,640x360,2.98 Mb/s,215.18 kb/s,Stereo
4,Last_man_on_earth_1964.mov,mov,prores,pcm_s16le,23.98 FPS,16:9,640x360,9.29 Mb/s,1536.00 kb/s,Stereo
5,The_Gun_and_the_Pulpit_formatOK.mp4,mp4,h264,aac,25.00 FPS,16:9,640x360,2.91 Mb/s,251.28 kb/s,Stereo
6,Voyage_to_the_Planet_of_Prehistoric_Women.mp4,mp4,hevc,mp3,29.97 FPS,16:9,640x360,8.04 Mb/s,320.00 kb/s,Stereo
7,The_Gun_and_the_Pulpit.avi,avi,rawvideo,pcm_s16le,25.00 FPS,720:404,720x404,87.44 Mb/s,1536.00 kb/s,Stereo
8,Cosmos_War_of_the_Planets.mp4,mp4,h264,aac,29.97 FPS,314:177,628x354,2.99 Mb/s,317.10 kb/s,Stereo
9,The_Hill_Gang_Rides_Again.mp4,mp4,h264,aac,25.00 FPS,16:9,640x360,7.54 Mb/s,253.27 kb/s,Stereo


### Compare before and after ###

In [115]:
def compare_original_and_converted(folder_path):
    """Compare metadata of original and converted videos."""
    original_files = [f for f in os.listdir(folder_path) if not f.endswith("_formatOK.mp4")]
    
    data = []
    
    for file in original_files:
        original_path = os.path.join(folder_path, file)
        converted_path = os.path.join(folder_path, file.rsplit(".", 1)[0] + "_formatOK.mp4")

        if os.path.exists(converted_path):
            original_metadata = get_video_metadata(original_path)
            converted_metadata = get_video_metadata(converted_path)

            if original_metadata and converted_metadata:
                orig_streams = original_metadata.get("streams", [])
                conv_streams = converted_metadata.get("streams", [])

                orig_video = next((s for s in orig_streams if s.get("codec_type") == "video"), {})
                conv_video = next((s for s in conv_streams if s.get("codec_type") == "video"), {})
                
                orig_audio = next((s for s in orig_streams if s.get("codec_type") == "audio"), {})
                conv_audio = next((s for s in conv_streams if s.get("codec_type") == "audio"), {})

                data.append({
                    "File Name": file,
                    "Orig Video Codec": orig_video.get("codec_name", "N/A"),
                    "Conv Video Codec": conv_video.get("codec_name", "N/A"),
                    "Orig Resolution": f"{orig_video.get('width', 'N/A')}x{orig_video.get('height', 'N/A')}",
                    "Conv Resolution": f"{conv_video.get('width', 'N/A')}x{conv_video.get('height', 'N/A')}",
                    "Orig Aspect Ratio": orig_video.get("display_aspect_ratio", "N/A"),
                    "Conv Aspect Ratio": conv_video.get("display_aspect_ratio", "N/A"),
                    "Orig Frame Rate": format_frame_rate(orig_video.get("avg_frame_rate", "N/A")),
                    "Conv Frame Rate": format_frame_rate(conv_video.get("avg_frame_rate", "N/A")),
                    "Orig Bit Rate (Video)": format_bit_rate(orig_video.get("bit_rate", "N/A"), "Mb"),
                    "Conv Bit Rate (Video)": format_bit_rate(conv_video.get("bit_rate", "N/A"), "Mb"),
                    "Orig Audio Codec": orig_audio.get("codec_name", "N/A"),
                    "Conv Audio Codec": conv_audio.get("codec_name", "N/A"),
                    "Orig Bit Rate (Audio)": format_bit_rate(orig_audio.get("bit_rate", "N/A"), "kb"),
                    "Conv Bit Rate (Audio)": format_bit_rate(conv_audio.get("bit_rate", "N/A"), "kb"),
                    "Orig Channels": orig_audio.get("channels", "N/A"),
                    "Conv Channels": conv_audio.get("channels", "N/A"),
                })
    
    df_comparison = pd.DataFrame(data)
    display(df_comparison)

# Run the comparison function
compare_original_and_converted("films")

,File Name,Orig Video Codec,Conv Video Codec,Orig Resolution,Conv Resolution,Orig Aspect Ratio,Conv Aspect Ratio,Orig Frame Rate,Conv Frame Rate,Orig Bit Rate (Video),Conv Bit Rate (Video),Orig Audio Codec,Conv Audio Codec,Orig Bit Rate (Audio),Conv Bit Rate (Audio),Orig Channels,Conv Channels
0,Last_man_on_earth_1964.mov,prores,h264,640x360,640x360,16:9,16:9,23.98 FPS,25.00 FPS,9.29 Mb/s,3.13 Mb/s,pcm_s16le,aac,1536.00 kb/s,240.47 kb/s,2,2
1,Voyage_to_the_Planet_of_Prehistoric_Women.mp4,hevc,h264,640x360,640x360,16:9,16:9,29.97 FPS,25.00 FPS,8.04 Mb/s,2.88 Mb/s,mp3,aac,320.00 kb/s,242.37 kb/s,2,2
2,The_Gun_and_the_Pulpit.avi,rawvideo,h264,720x404,640x360,N/A,16:9,25.00 FPS,25.00 FPS,87.44 Mb/s,2.91 Mb/s,pcm_s16le,aac,1536.00 kb/s,251.28 kb/s,2,2
3,Cosmos_War_of_the_Planets.mp4,h264,h264,628x354,640x360,314:177,16:9,29.97 FPS,25.00 FPS,2.99 Mb/s,2.97 Mb/s,aac,aac,317.10 kb/s,245.65 kb/s,2,2
4,The_Hill_Gang_Rides_Again.mp4,h264,h264,640x360,640x360,16:9,16:9,25.00 FPS,25.00 FPS,7.54 Mb/s,2.98 Mb/s,aac,aac,253.27 kb/s,215.18 kb/s,2,2
